# Colab Runner: dataset-specific heads
Train-derived compact heads, validation, then held-out test. Source labels stay unchanged.
Run the CPU smoke first. Full retraining is opt-in and needs a fresh runs directory after the resplit.


In [ ]:
# v4 bootstrap: mount Drive, install pinned deps, verify cleaned data
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or os.path.isdir("/content")
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

REPO = Path("/content/minor-project") if IN_COLAB else Path.cwd()
if not (REPO / "training" / "run.py").exists():
    # adjust to your Drive copy of the repo
    REPO = Path("/content/drive/MyDrive/object-detection/minor-project")
os.chdir(REPO)
print("repo:", REPO)

# single installer entry (idempotent, W8.1)
subprocess.check_call([sys.executable, "install.py"])

# environment report - paste this into any bug report (W8.5)
subprocess.check_call([sys.executable, "training/run.py", "--env"])

# verify cleaned data; if missing, run the cleaner first:
#   python data_cleaning/clean_build_v4.py --help
DATA = Path("/content/drive/MyDrive/object-detection/cleaned_data") if IN_COLAB else REPO / "data" / "cleaned_data"
for ds in ("codrone", "visdrone", "dota"):
    ok = (DATA / ds / "train" / "images").is_dir()
    print(f"{ds:<10}: {'ready' if ok else 'MISSING - run data_cleaning/clean_build_v4.py'}")


## Smoke test (optional, ~minutes)
Verifies the install end-to-end on generated mock data before burning GPU time.


In [ ]:
# CPU smoke across all three engine families, including exports and report figures
!python training/run.py --models custom-obb ao2-detr yolov8n-obb --datasets mock --steps train,eval,summary,export,report --test --device cpu --no-auto-resume --runs-dir /content/class_head_smoke


## Full benchmark


In [ ]:
# Explicit opt-in. Do not use old runs/checkpoints after a dataset resplit.
RUN_FULL_BENCHMARK = False
if RUN_FULL_BENCHMARK:
    subprocess.check_call([sys.executable, "training/run.py", "--models", "all", "--datasets", "all", "--mode", "full", "--data-dir", str(DATA), "--no-auto-resume", "--runs-dir", "/content/drive/MyDrive/object-detection/runs_class_heads_fresh"])
else:
    print("Full benchmark disabled. Review dataset paths and runtime before opting in.")


## Single model / subset examples
`--models yolo26`, `--models nano`, `--models dense`, or any list. `--order bigfirst` inverts the small-first default.


In [ ]:
# Example only; enable explicitly after checking runtime and dataset path.
RUN_SUBSET = False
if RUN_SUBSET:
    subprocess.check_call([sys.executable, "training/run.py", "--models", "lsknet", "--datasets", "codrone", "--data-dir", str(DATA), "--runs-dir", "/content/drive/MyDrive/object-detection/runs_class_heads_subset_fresh", "--no-auto-resume"])
